<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/06_first_ratings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
rating_sheet = pd.read_csv(folder / "rating_sheet.csv", dtype=str, keep_default_na=False)
my_ratings_path = folder / "my_ratings.csv"

if my_ratings_path.exists():
    my_ratings = pd.read_csv(my_ratings_path, dtype=str, keep_default_na=False)
else:
    my_ratings = pd.DataFrame(columns=["file_id", "rating"])

print(len(my_ratings), "rows already rated")

30 rows already rated


In [ ]:
rows_to_rate = 30

for i in range(rows_to_rate):
    file_id = rating_sheet.loc[i, "file_id"]
    if file_id in list(my_ratings["file_id"]):
        continue

    print("row", i + 1, "of", rows_to_rate)
    print("correct text:", rating_sheet.loc[i, "correct_text"])
    print("model text:  ", rating_sheet.loc[i, "model_text"])
    answer = input("1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: ")

    if answer == "q":
        break
    if answer not in ["1", "0", "?"]:
        print("wrong key, this row is skipped. Run this cell again to rate it.")
        continue

    my_ratings.loc[len(my_ratings)] = [file_id, answer]
    my_ratings.to_csv(my_ratings_path, index=False)
    print()

In [ ]:
print(len(my_ratings), "rows rated")
print(my_ratings["rating"].value_counts())

30 rows rated
rating
0    25
1     5
Name: count, dtype: int64


In [ ]:
scores = pd.read_csv(folder / "scores_200.csv", dtype={"file_id": str})

unsure_count = (my_ratings["rating"] == "?").sum()

rated = my_ratings[my_ratings["rating"].isin(["0", "1"])].copy()
rated = rated.merge(scores[["file_id", "wer", "cer"]], on="file_id")
rated["rating"] = rated["rating"].astype(int)

print("rated rows:", len(rated), "| unsure rows:", unsure_count)

rated rows: 30 | unsure rows: 0


In [ ]:
rated.groupby("rating")[["wer", "cer"]].mean().round(2)

,wer,cer
rating,,
0,0.83,0.25
1,0.48,0.06


In [ ]:
print("WER and rating:", round(rated["rating"].corr(rated["wer"], method="spearman"), 2))
print("CER and rating:", round(rated["rating"].corr(rated["cer"], method="spearman"), 2))

WER and rating: -0.3
CER and rating: -0.49
